# Smartphone ML Intelligence

## Part 6 — Specification-Based Value Analysis

### Objective

Estimate how each smartphone's actual market price compares with the price expected from its specifications.

The analysis uses out-of-fold price predictions so that each smartphone is evaluated by a model that was not trained on that smartphone.

This allows us to investigate:

- smartphones priced below their specification-based expectation
- smartphones carrying a price premium relative to their specifications
- value patterns across brands and market segments
- whether certain parts of the market systematically offer stronger specification value

The resulting value signals will later be incorporated into the recommendation engine.

### 1. Importing & Loading

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

In [2]:
DATA_PATH = Path("../data/processed/phones_clean.csv")

phones = pd.read_csv(
    DATA_PATH,
    parse_dates=["launch_date"]
)

print(f"Rows: {phones.shape[0]}")
print(f"Columns: {phones.shape[1]}")

phones.head()

Rows: 278
Columns: 48


,name,brand,sale,processor_type,processor,fab,ram_type,rom_type,display_type,size,...,speaker,wireless_charging,ip_rating,price,avaibility,has_ultrawide,has_third_camera,has_fourth_camera,launch_year,phone_age_days
0,CMF Phone 1,Nothing,Online,Mediatek,Dimensity 7300,4,LPDDR4X,UFS 2.2,Amoled,6.67,...,Mono,False,52,16000,0,False,True,False,2024,806
1,CMF Phone 2 Pro,Nothing,Online,Mediatek,Dimensity 7300,4,LPDDR4X,UFS 2.2,Amoled,6.77,...,Mono,False,54,19000,0,True,True,False,2025,482
2,Galaxy A07,Samsung,Offline,Mediatek,Dimensity 6300,6,LPDDR4X,UFS 2.2,IPS LCD,6.70,...,Mono,False,54,20000,1,False,True,False,2026,229
3,Galaxy A17,Samsung,Offline,Exynos,1330,5,LPDDR4X,UFS 2.2,Amoled,6.70,...,Mono,False,54,28000,1,True,True,False,2025,389
4,Galaxy A27,Samsung,Offline,Snapdragon,6 Gen 3,4,LPDDR5X,UFS 3.1,Amoled,6.70,...,Mono,False,64,34000,1,True,True,False,2026,89


### 2. Defining Features & Target

In [3]:
excluded_features = [
    # Target
    "price",

    # Identifiers / descriptive text
    "name",
    "launch_date",
    "colors",
    "camera_sensor_name",

    # Avoid learning exact device / chipset identity
    "processor",

    # Benchmarks
    "antutu",
    "geekbench_single",
    "geekbench_multiple",

    # Market-position information
    "sale"
]

spec_features = [
    column
    for column in phones.columns
    if column not in excluded_features
]

X = phones[spec_features].copy()
y = phones["price"].copy()

print("Number of features:", len(spec_features))
print(spec_features)


Number of features: 38
['brand', 'processor_type', 'fab', 'ram_type', 'rom_type', 'display_type', 'size', 'curved_display', 'refresh_rate', 'hbm', 'protection', 'ppi', 'battery', 'charging_speed', 'thickness', 'back_material', 'frame_material', 'weight', 'camera_number', 'useful_camera_number', 'main', 'ultrawide', 'third', 'third_mp', 'fourth', 'fourth_mp', 'rear_video', 'front', 'front_video', 'speaker', 'wireless_charging', 'ip_rating', 'avaibility', 'has_ultrawide', 'has_third_camera', 'has_fourth_camera', 'launch_year', 'phone_age_days']


### 3. Identifying Features

In [4]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['fab', 'size', 'refresh_rate', 'hbm', 'ppi', 'battery', 'charging_speed', 'thickness', 'weight', 'camera_number', 'useful_camera_number', 'main', 'ultrawide', 'third_mp', 'fourth_mp', 'front', 'ip_rating', 'avaibility', 'launch_year', 'phone_age_days']

Categorical features:
['brand', 'processor_type', 'ram_type', 'rom_type', 'display_type', 'curved_display', 'protection', 'back_material', 'frame_material', 'third', 'fourth', 'rear_video', 'front_video', 'speaker', 'wireless_charging', 'has_ultrawide', 'has_third_camera', 'has_fourth_camera']


### 4. Preprocessing

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

### 5. Building Price Model

In [6]:
from sklearn.ensemble import GradientBoostingRegressor

price_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=2,
    min_samples_leaf=3,
    random_state=42
)

price_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", price_model)
])

### 6. Generating repeated Out-of-Fold predictions

In [7]:
import numpy as np

from sklearn.base import clone
from sklearn.model_selection import RepeatedKFold

cv = RepeatedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

oof_predictions = np.zeros(len(phones))
prediction_counts = np.zeros(len(phones))

for train_idx, test_idx in cv.split(X):

    model = clone(price_pipeline)

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    oof_predictions[test_idx] += predictions
    prediction_counts[test_idx] += 1

oof_predictions = (
    oof_predictions /
    prediction_counts
)

### 7. Evaluating Estimates

In [8]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    y,
    oof_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y,
        oof_predictions
    )
)

r2 = r2_score(
    y,
    oof_predictions
)

print(f"OOF MAE  : ₹{mae:,.0f}")
print(f"OOF RMSE : ₹{rmse:,.0f}")
print(f"OOF R²   : {r2:.3f}")

OOF MAE  : ₹7,613
OOF RMSE : ₹12,063
OOF R²   : 0.882


### 8. Actual Value Analysis

In [9]:
value_analysis = phones.copy()

value_analysis["expected_price"] = (
    oof_predictions
)

value_analysis["value_gap"] = (
    value_analysis["expected_price"]
    - value_analysis["price"]
)

value_analysis["value_gap_pct"] = (
    value_analysis["value_gap"]
    / value_analysis["price"]
) * 100

In [10]:
value_analysis[
    [
        "name",
        "price",
        "expected_price",
        "value_gap",
        "value_gap_pct"
    ]
].sort_values(
    "value_gap_pct",
    ascending=False
)

,name,price,expected_price,value_gap,value_gap_pct
133,Oneplus Nord CE 4,25000,57253.314707,32253.314707,129.013259
91,Moto Edge 70 Max,55000,102105.160858,47105.160858,85.645747
149,Oppo K13,13000,22757.452230,9757.452230,75.057325
61,Iqoo Z10 Lite,10000,16721.975579,6721.975579,67.219756
54,Iqoo 13,55000,90126.955655,35126.955655,63.867192
...,...,...,...,...,...
272,Xiaomi 14,70000,47453.780118,-22546.219882,-32.208886
271,Vivo X300 Ultra,160000,103910.867329,-56089.132671,-35.055708
242,Tecno Pova 8 Pro,50000,30494.105932,-19505.894068,-39.011788
116,Nothing Phone 4b,40000,24252.188605,-15747.811395,-39.369528


In [41]:
result = (
    value_analysis[
        [
            "name",
            "price",
            "expected_price",
            "value_gap",
            "value_gap_pct"
        ]
    ]
    .sort_values(
        "value_gap_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

with pd.option_context("display.max_rows", None):
    display(result)

,name,price,expected_price,value_gap,value_gap_pct
0,Oneplus Nord CE 4,25000,57253.314707,32253.314707,129.013259
1,Moto Edge 70 Max,55000,102105.160858,47105.160858,85.645747
2,Oppo K13,13000,22757.452230,9757.452230,75.057325
3,Iqoo Z10 Lite,10000,16721.975579,6721.975579,67.219756
4,Iqoo 13,55000,90126.955655,35126.955655,63.867192
5,Moto Edge 50 Neo,24000,39267.036090,15267.036090,63.612650
6,Lava Play Ultra,14000,22856.419774,8856.419774,63.260141
7,Poco C81x,11500,17146.528305,5646.528305,49.100246
8,Lava Bold N2,13000,19324.433511,6324.433511,48.649489
9,Galaxy S24 Snapdragon,40000,58925.617976,18925.617976,47.314045
